# 06 — Data Format Converter: XML (Pascal VOC) → YOLO

Converts bounding-box annotations from Pascal VOC XML format to YOLO `.txt` format and builds the `data.yaml` required by YOLOv8.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, shutil, random, yaml
import xml.etree.ElementTree as ET
import cv2

print('Libraries imported successfully.')

Libraries imported successfully.


In [ ]:
# ── EDIT THESE PATHS TO MATCH YOUR DRIVE STRUCTURE ──────────────────────────
XML_DIR      = '/content/drive/MyDrive/ROI_Endoscopy_Paper/annotations/xml'
IMG_DIR      = '/content/drive/MyDrive/ROI_Endoscopy_Paper/annotations/images'
YOLO_DATA_DIR = '/content/drive/MyDrive/ROI_Endoscopy_Paper/yolo_dataset'
# ─────────────────────────────────────────────────────────────────────────────

for split in ['train', 'val']:
    os.makedirs(os.path.join(YOLO_DATA_DIR, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DATA_DIR, 'labels', split), exist_ok=True)

print(f'XML dir   : {XML_DIR}')
print(f'Image dir : {IMG_DIR}')
print(f'YOLO dir  : {YOLO_DATA_DIR}')

XML dir   : /content/drive/MyDrive/ROI_Endoscopy_Paper/annotations/xml
Image dir : /content/drive/MyDrive/ROI_Endoscopy_Paper/annotations/images
YOLO dir  : /content/drive/MyDrive/ROI_Endoscopy_Paper/yolo_dataset


In [ ]:
# ── EDIT CLASS NAMES TO MATCH YOUR ANNOTATIONS ───────────────────────────────
CLASSES = {
    'polyp':        0,
    'lesion':       1,
    'inflammation': 2
}
# ─────────────────────────────────────────────────────────────────────────────

print(f'Classes : {CLASSES}')

Classes : {'polyp': 0, 'lesion': 1, 'inflammation': 2}


In [ ]:
def get_image_dimensions(img_path):
    img = cv2.imread(img_path)
    if img is not None:
        h, w = img.shape[:2]
        return w, h
    return None, None


def convert_xml_to_yolo(xml_file, img_width, img_height):
    """
    Returns a list of YOLO annotation strings:
      class_id  cx  cy  w  h   (all values normalised 0‑1)
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()
    lines = []

    for obj in root.findall('object'):
        label = obj.find('name').text
        if label not in CLASSES:
            print(f'  [skip] unknown class "{label}" in {os.path.basename(xml_file)}')
            continue

        b = obj.find('bndbox')
        xmin = float(b.find('xmin').text)
        ymin = float(b.find('ymin').text)
        xmax = float(b.find('xmax').text)
        ymax = float(b.find('ymax').text)

        cx = max(0.0, min(1.0, (xmin + xmax) / 2.0 / img_width))
        cy = max(0.0, min(1.0, (ymin + ymax) / 2.0 / img_height))
        bw = max(0.0, min(1.0, (xmax - xmin) / img_width))
        bh = max(0.0, min(1.0, (ymax - ymin) / img_height))

        lines.append(f'{CLASSES[label]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')

    return lines


print('Conversion functions defined.')

Conversion functions defined.


In [ ]:
random.seed(42)

xml_files = sorted([f for f in os.listdir(XML_DIR) if f.endswith('.xml')])
random.shuffle(xml_files)

split_idx   = int(0.8 * len(xml_files))
train_files = xml_files[:split_idx]
val_files   = xml_files[split_idx:]

print(f'Total XML files : {len(xml_files)}')
print(f'Train           : {len(train_files)}')
print(f'Val             : {len(val_files)}')

Total XML files : 239
Train           : 191
Val             : 48


In [ ]:
def process_dataset(file_list, split_name):
    converted, skipped = 0, 0

    for xml_name in file_list:
        img_name = xml_name.replace('.xml', '.jpg')
        xml_path = os.path.join(XML_DIR, xml_name)
        img_path = os.path.join(IMG_DIR, img_name)

        if not os.path.exists(img_path):
            print(f'  [warn] image not found: {img_name}')
            skipped += 1
            continue

        w, h = get_image_dimensions(img_path)
        if w is None:
            skipped += 1
            continue

        annotations = convert_xml_to_yolo(xml_path, w, h)
        if not annotations:
            skipped += 1
            continue

        # Copy image
        shutil.copy(img_path, os.path.join(YOLO_DATA_DIR, 'images', split_name, img_name))

        # Write YOLO label
        txt_name = img_name.replace('.jpg', '.txt')
        with open(os.path.join(YOLO_DATA_DIR, 'labels', split_name, txt_name), 'w') as f:
            f.write('\n'.join(annotations))

        converted += 1

    print(f'{split_name:>5}: {converted} converted, {skipped} skipped')
    return converted


print('Converting train split...')
n_train = process_dataset(train_files, 'train')

print('Converting val split...')
n_val = process_dataset(val_files, 'val')

print(f'\nTotal converted: {n_train + n_val}')

Converting train split...
  [skip] unknown class "normal" in 11e01bb3-b9b7-466d-9187-13a60daef108.avi_frame_00004.xml
  [skip] unknown class "normal" in c205ec73-4652-4f91-8711-ddb6a50e1d16.avi_frame_00013.xml
  [skip] unknown class "normal" in 7fe9e5ab-4e01-44f0-9139-604649b237e9.avi_frame_00000.xml
  [skip] unknown class "normal" in a75bc3f1-949c-4629-bc66-c31c139be6df.avi_frame_00012.xml
  [skip] unknown class "normal" in 11e01bb3-b9b7-466d-9187-13a60daef108.avi_frame_00008.xml
  [skip] unknown class "normal" in 7fe9e5ab-4e01-44f0-9139-604649b237e9.avi_frame_00007.xml
  [skip] unknown class "normal" in f1dfa843-1e53-4a5d-b3b1-d29dc08e968a.avi_frame_00015.xml
  [skip] unknown class "normal" in 1c1ca45a-315e-4ad0-9343-2425d2faf648.avi_frame_00003.xml
  [skip] unknown class "normal" in 17241dcf-de85-461d-9435-f5134035aacb.avi_frame_00011.xml
  [skip] unknown class "normal" in 17241dcf-de85-461d-9435-f5134035aacb.avi_frame_00000.xml
  [skip] unknown class "normal" in da5d2629-3a74-4ec0-

In [ ]:
yaml_content = {
    'path' : YOLO_DATA_DIR,
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : len(CLASSES),
    'names': {v: k for k, v in CLASSES.items()}
}

yaml_path = os.path.join(YOLO_DATA_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print(f'data.yaml saved to: {yaml_path}\n')
with open(yaml_path) as f:
    print(f.read())

data.yaml saved to: /content/drive/MyDrive/ROI_Endoscopy_Paper/yolo_dataset/data.yaml

names:
  0: polyp
  1: lesion
  2: inflammation
nc: 3
path: /content/drive/MyDrive/ROI_Endoscopy_Paper/yolo_dataset
train: images/train
val: images/val



In [ ]:
for split in ['train', 'val']:
    imgs = len(os.listdir(os.path.join(YOLO_DATA_DIR, 'images', split)))
    lbls = len(os.listdir(os.path.join(YOLO_DATA_DIR, 'labels', split)))
    status = '✅' if imgs == lbls else '⚠️  MISMATCH'
    print(f'{split:>5}: {imgs} images, {lbls} labels  {status}')

print('\nSample YOLO annotation:')
first_txt = os.listdir(os.path.join(YOLO_DATA_DIR, 'labels', 'train'))[0]
with open(os.path.join(YOLO_DATA_DIR, 'labels', 'train', first_txt)) as f:
    print(f.read())
print('Format: class_id  cx  cy  width  height  (normalised 0‑1)')

train: 57 images, 57 labels  ✅
  val: 13 images, 13 labels  ✅

Sample YOLO annotation:
0 0.606257 0.429115 0.642903 0.660069
Format: class_id  cx  cy  width  height  (normalised 0‑1)
